In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
from torch.utils.data import WeightedRandomSampler
from fastai.callback.tracker import SaveModelCallback, Recorder
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import torch
from mtrain.neg_mask.model.datasets.blur_pad_dl import random_tfm, BlurPadDataset
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import show, mkdir, DiskImage, DiskBooleanMask
from pytorch_grad_cam import (
    GradCAM,
)
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from tqdm import tqdm
from mtrain.neg_mask.model.show import (
    get_preds_for_ds,
    show_classification_report,
    show_confusion_matrix,
    show_confusion_matrix_using_preds,
)
from mtrain.neg_mask.model.datasets.blur_pad_dl import CropTfmsOutsideBbox
from functools import partial
from sklearn.model_selection import train_test_split
from fastai.basics import DataLoaders, default_device
from mtrain.denorm import denormalize_imagenet, denormalize_4chan_imagenet
from mtrain.utils import show, it_chain
from fastai.callback.all import ProgressCallback
from fastai.basics import F1Score, Precision, Recall, CrossEntropyLossFlat
from fastai.vision.all import vision_learner, xresnet18

In [ ]:
from mtrain.utils import globL, mkdir

# CLEAN_PATH = Path(
#     "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/train"
# )
# FOVEATED_PATH = Path(
#     "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/foveated"
# )
FOVEATED_DS_PATH = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/training/dataset"
)
# UNIT_TEST_DEST = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/unit-tests/negmask")
# mkdir(UNIT_TEST_DEST / "train")
# mkdir(UNIT_TEST_DEST / "masks")

# images = globL(CLEAN_PATH / "train", "*.jpg")

# import shutil
# images = images[:256]
# for img in images:
#     mask = CLEAN_PATH / "masks" / f"{img.stem}.png"

#     shutil.copy(img, UNIT_TEST_DEST / "train" / img.name)
#     shutil.copy(mask, UNIT_TEST_DEST / "masks" / mask.name)

DS_PATH = FOVEATED_DS_PATH

In [ ]:
def get_learner(dls):
    learn = vision_learner(
        dls,
        xresnet18,
        metrics=[F1Score(average="macro"), Precision(), Recall()],
        loss_func=CrossEntropyLossFlat(CLS_WEIGHT),
        n_out=2,
        normalize=False,
        n_in=3,
        pretrained=True,
    )
    learn.remove_cb(Recorder)
    learn.remove_cb(SaveModelCallback)
    learn.add_cb(Recorder())
    learn.add_cbs([SaveModelCallback(monitor="f1_score", fname="best")])
    learn = learn.remove_cb(ProgressCallback)
    return learn


def get_denormalized(tens):
    image, mask = None, None
    image = denormalize_imagenet(tens)
    image = image.permute([1, 2, 0]).numpy()
    if mask is not None:
        mask = mask.numpy()
    return image, mask


def show_gradcam_for_image(
    learn, input_tensor, target_label_idx=None, layer_name="0.7.1.conv1"
):
    target_layers = [learn.model.get_submodule(layer_name)]
    img_arr, _ = get_denormalized(input_tensor[0])

    targets = [ClassifierOutputTarget(target_label_idx)]

    with GradCAM(model=learn.model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]
        print(img_arr.shape, grayscale_cam.shape)
        visualization = show_cam_on_image(img_arr, grayscale_cam, use_rgb=True)
        model_outputs = cam.outputs

        return visualization, img_arr, model_outputs


def show_reports(learner):
    preds = learner.get_preds(dl=learner.dls.valid, with_decoded=True, with_loss=True)
    probs, targs, decoded, losses = preds
    labels = list(BlurPadDataset.LABEL_BY_IDX.keys())
    show_classification_report(probs, targs, labels)
    show_confusion_matrix_using_preds(probs, targs, labels)
    return probs, targs, decoded, losses

In [ ]:
CLS_WEIGHT = torch.tensor([1.0, 1.0]).float().to("mps")


def get_weight(p, taco_weight=1.0, manual_weight=1.5, default_weight=1.0):
    p = Path(p)
    if "taco" in p.name:
        return taco_weight
    if "manual" in p.name:
        return manual_weight
    else:
        return default_weight


def get_dls(
    num_samples,
    tfm,
    crop_size=224,
    ds_path=DS_PATH,
    min_area=35,
    min_bbox_length=3,
    max_area=None,
    path_filter=None,
):

    image_paths = list((ds_path / "train").glob("*.jpg"))[:num_samples]
    if path_filter is not None:
        image_paths = list(filter(path_filter, image_paths))
    stratify = [BlurPadDataset.label_func(p) for p in image_paths]
    train_paths, valid_paths = train_test_split(
        image_paths, test_size=0.2, stratify=stratify, random_state=42
    )

    train_ds = BlurPadDataset(
        train_paths,
        ds_path / "masks",
        crop_size,
        False,
        crop_mutator=tfm,
        bbox_pad=3,
        min_area=min_area,
        min_bbox_length=min_bbox_length,
        max_area=max_area,
    )
    valid_ds = BlurPadDataset(
        valid_paths,
        ds_path / "masks",
        crop_size,
        True,
        crop_mutator=tfm,
        bbox_pad=3,
        min_area=min_area,
        min_bbox_length=min_bbox_length,
        max_area=max_area,
    )
    image_paths = train_ds.image_paths
    train_weights = [get_weight(p) for p in image_paths]
    train_sampler = WeightedRandomSampler(
        weights=train_weights, num_samples=len(image_paths), replacement=True
    )
    dls = DataLoaders.from_dsets(
        train_ds,
        valid_ds,
        device=default_device(),
        num_workers=4,
        bs=16,
        # pin_memory=True,
        persistent_workers=True,
        dl_kwargs=[
            {"sampler": train_sampler, "shuffle": False},
            {"shuffle": False},  # Validation defaults
        ],
    )  # don't respawn workers each epoch)
    return dls


def vis_sample_ds(ds, idx):
    tens, targ = ds[idx]
    print("target", targ)
    print("shape", tens.shape)
    img, _ = get_denormalized(tens)
    plt.imshow(img, cmap="gray")
    plt.show()


def vis_sample(dls, idx):
    vis_sample_ds(dls.train_ds, idx)

In [ ]:
# to counter the problem of the model focusing on texture/noise
# we decrease the probability of adding noise with each sweep while maintaining accuracy
# the next step is to remove overwrite noise
# then next is decreasing the add noise frequency
# first i would need to seee the performance of the model
#  on different types of aux transforms (step down? gaussian? blur?)
# our final model has no noise, and one kind of step down function
# we need to test it on all transforms and find the winner
# for each we do successive training by decreasing the add_noise chance parameter
def blur_tfm(
    cropped_image, mask, inner_bbox, add_noise_chance, blur_kernel_sz, blur_sigma
):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.overwrite_with_blur(blur_kernel_sz, blur_sigma)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop


def step_down_tfm(cropped_image, mask, inner_bbox, add_noise_chance, ratio):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down(ratio)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop, mask


def step_down_gauss_tfm(cropped_image, mask, inner_bbox, add_noise_chance, min_value):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down_gaussian(min_value)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop

In [ ]:
def get_initialised_learner():
    dls = get_dls(100, random_tfm)
    learner = get_learner(dls)
    MODELS_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models")
    path = (
        MODELS_DIR
        / "negmask"
        / "tfm-stepedge_data-withusefultaco_iter-35_arch-xresnet18.pth"
    )
    state_dict = torch.load(path)
    learner.model.load_state_dict(state_dict)
    return learner

In [ ]:
st_ed_tfm = partial(step_down_tfm, ratio=0.5)

In [ ]:
st_ed_learner = get_initialised_learner()

# NEGMASK_224_STEP_EDGE = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/st_ed_tfm0-final-all-data-iter-20.pth"
# NEG_MASKSTEP_EDGE_MODEL_PATH = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-unblur/tfm-stepdown_ratio-5_samples-all-xresnet18_iter-20.pth"
# NEGMASK_224_STEP_EDGE = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/negmask/tfm-stepedge_data-withtaco_iter-25_arch-xresnet18.pth"
# NEGMASK_224_STEP_EDGE = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/tfm-stepedge_data-withusefultaco_iter-35_arch-xresnet18.pth"
# FOVEATED_NEGMASK_WITH_WALLS = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/foveated-224/iter-30-v2-with-walls-xresnet18.pth"
# PHASE_2_STEP_EDGE_MODEL_PATH = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-unblur/tfm-stepdown_ratio-5_samples-all-xresnet18_iter-20.pth"

In [ ]:
from mtrain.utils import show_negmask_ds

show_negmask_ds(DS_PATH, 4)

In [ ]:
# def no_mapi_walls(path):
#     is_wall_mapi = "walls-mapillary" in Path(path).stem
#     return not is_wall_mapi


st_ed_tfm0 = partial(st_ed_tfm, add_noise_chance=-1)
dls = get_dls(200000, st_ed_tfm0)

In [ ]:
st_ed_learner.dls = dls

In [ ]:
st_ed_learner.eval()
res = show_reports(st_ed_learner)

In [ ]:
vis_sample(dls, 5)

In [ ]:
# st_ed_learner.dls = dls

st_ed_learner.train()
st_ed_learner.freeze()
st_ed_learner.fit_one_cycle(4)

In [ ]:
st_ed_learner.unfreeze()

In [ ]:
st_ed_learner.fit_one_cycle(6)

In [ ]:
SUCC_MODELS_DIR = mkdir(
    Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models/foveated-224")
)

torch.save(
    st_ed_learner.model.state_dict(),
    SUCC_MODELS_DIR / "iter-12-xresnet18.pth",
)

In [ ]:
st_ed_learner.eval()
res = show_reports(st_ed_learner)

In [ ]:
st_ed_learner.train()
st_ed_learner.lr_find()

In [ ]:
st_ed_learner.fit_one_cycle(6, lr_max=3.0199516913853586e-05)

In [ ]:
st_ed_learner.eval()
res = show_reports(st_ed_learner)

In [ ]:
_ = st_ed_learner.eval()

In [ ]:
old_learner = get_initialised_learner()
old_learner.model.load_state_dict(torch.load(SUCC_MODELS_DIR / "iter-6-xresnet18.pth"))
_ = old_learner.eval()

In [ ]:
new_preds = st_ed_learner.get_preds(dl=st_ed_learner.dls.valid, with_decoded=True, with_loss=True)
_, new_targs, new_decoded, _ = new_preds

In [ ]:
old_preds = old_learner.get_preds(dl=st_ed_learner.dls.valid, with_decoded=True, with_loss=True)
_, old_targs, old_decoded, _ = old_preds

In [ ]:
diff_idxs = [i for (i, is_eq) in enumerate(new_decoded == old_decoded) if not is_eq]

In [ ]:
i = 28
idx = diff_idxs[i]
print("new", new_decoded[idx])
print("old", old_decoded[idx])
vis_sample_ds(st_ed_learner.dls.valid_ds, idx)

In [ ]:
# st_ed_learner.unfreeze()
st_ed_learner.fit_one_cycle(5)


In [ ]:
st_ed_learner.eval()
res = show_reports(st_ed_learner)

In [ ]:
from fastai.callback.tracker import SaveModelCallback, Recorder


st_ed_learner.remove_cb(Recorder)
st_ed_learner.remove_cb(SaveModelCallback)
st_ed_learner.add_cb(Recorder())
st_ed_learner = st_ed_learner.add_cbs(
    [SaveModelCallback(monitor="f1_score", fname="best")]
)

In [ ]:
st_ed_learner.cbs

In [ ]:
st_ed_learner.recorder.metric_names

In [ ]:
st_ed_learner.cbs = st_ed_learner.cbs[:2]

In [ ]:
st_ed_learner.unfreeze()
st_ed_learner.fit_one_cycle(10)

In [ ]:
from fastai.vision.all import valley

st_ed_learner.lr_find()

In [ ]:
st_ed_learner.fit_one_cycle(15, lr_max=1e-3)

In [ ]:
SUCC_MODELS_DIR = mkdir(
    Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models/foveated-224")
)

torch.save(
    st_ed_learner.model.state_dict(),
    SUCC_MODELS_DIR / "iter-30-v2-with-walls-xresnet18.pth",
)

In [ ]:
# st_ed_learner.unfreeze()
st_ed_learner.fit_one_cycle(2)


In [ ]:
st_ed_learner.fit_one_cycle(4, lr_max=1e-4)

In [ ]:
res = show_reports(st_ed_learner)

In [ ]:
probs, targs, decoded, losses = res
sorted_losses = list(reversed(sorted([(loss, i) for i, loss in enumerate(losses)])))
top_loss_idxs = [sl[1] for sl in sorted_losses]
min_losses = [sl[1] for sl in reversed(sorted_losses)]


In [ ]:
vis_sample_ds(st_ed_learner.dls.valid_ds, top_loss_idxs[15])

In [ ]:
# lets try bumping it out?
st_ed_learner.fit_one_cycle(6, lr_max=1e-3)

In [ ]:
SUCC_MODELS_DIR = mkdir(
    Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models/foveated-224")
)

torch.save(
    st_ed_learner.model.state_dict(),
    SUCC_MODELS_DIR / "iter-6-xresnet18.pth",
)

In [ ]:
res = show_reports(st_ed_learner)

In [ ]:
probs, targs, decoded, losses = res
sorted_losses = list(reversed(sorted([(loss, i) for i, loss in enumerate(losses)])))
top_loss_idxs = [sl[1] for sl in sorted_losses]
min_losses = [sl[1] for sl in reversed(sorted_losses)]


In [ ]:
to_remove = set([])

In [ ]:
preds = st_ed_learner.get_preds(
    dl=st_ed_learner.dls.valid, with_decoded=True, with_loss=True
)
probs, targs, decoded, losses = preds


In [ ]:
# first we find the stuff that was marked as trash but was other

# is_other_pred_trash_idx  = torch.where((decoded == 1) & (targs == 0))[0]

is_trash_pred_other_idx = torch.where((decoded == 0) & (targs == 1))[0]
len(is_other_pred_trash_idx), len(is_trash_pred_other_idx)

In [ ]:
probs, targs, decoded, losses = res
sorted_losses = list(reversed(sorted([(loss, i) for i, loss in enumerate(losses)])))
top_loss_idxs = [sl[1] for sl in sorted_losses]
min_losses = [sl[1] for sl in reversed(sorted_losses)]

In [ ]:
# idx = top_loss_idxs[31]
idx = is_other_pred_trash_idx[12]
ds = st_ed_learner.dls.valid_ds
current_path = ds.image_paths[idx]
print(current_path.name)
vis_sample_ds(ds, idx)

In [ ]:
print("adding", current_path.name)
to_remove.add(current_path)

In [ ]:
for rm in to_remove:
    mdir = rm.parent.parent / "masks"
    mpath = mdir / f"{rm.stem}.png"
    # print(rm, mpath)
    rm.unlink()
    mpath.unlink()

In [ ]:
to_remove

In [ ]:
idx = top_loss_idxs[18]
vis_sample_ds(st_ed_learner.dls.valid_ds, idx)

In [ ]:
res = show_reports(st_ed_learner)

In [ ]:
st_ed_tfm0 = partial(st_ed_tfm, add_noise_chance=-1)
dls = get_dls(20000, st_ed_tfm0)

In [ ]:
st_ed_learner.dls = dls

st_ed_learner.freeze()
st_ed_learner.fit_one_cycle(3)

In [ ]:
st_ed_learner.unfreeze()
st_ed_learner.fit_one_cycle(4)

In [ ]:
torch.save(
    st_ed_learner.model.state_dict(),
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/st_ed_tfm0-final-all-data-with-taco-iter-25.pth",
)


In [ ]:
show_reports(st_ed_learner)

In [ ]:
st_ed_learner.fit_one_cycle(5, lr_max=slice(1e-5, 1e-4))

In [ ]:
torch.save(
    st_ed_learner.model.state_dict(),
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/st_ed_tfm0-final-all-data-with-taco-iter-15.pth",
)


In [ ]:
st_ed_learner.loss_func = CrossEntropyLossFlat()

In [ ]:
st_ed_learner.unfreeze()
st_ed_learner.fit_one_cycle(5)

In [ ]:
st_ed_learner.lr_find()

In [ ]:
res = show_reports(st_ed_learner)

In [ ]:
torch.save(
    st_ed_learner.model.state_dict(),
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/st_ed_tfm0-final-all-data-with-taco-iter-10.pth",
)

In [ ]:
idx = top_loss_idxs[6]
viz, img, mo = show_gradcam_for_image(
    st_ed_learner,
    st_ed_learner.dls.valid_ds[idx][0].unsqueeze(0),
    1,
    "0.7.1.convpath.1.0",
)
print(mo)
show([viz, img])


I will need to train with more noise, lesser data first

In [ ]:
dls = get_dls(500, random_tfm)
learner = get_learner(dls)

In [ ]:
# learner.fine_tune(1)
learner.fit_one_cycle(20)

In [ ]:
dls = get_dls(3000, random_tfm)
learner.dls = dls

In [ ]:
learner.freeze()
learner.fit_one_cycle(4)
learner.unfreeze()
learner.fit_one_cycle(10)

In [ ]:
res = show_reports(learner)
probs, targs, decoded, losses = res
sorted_losses = list(reversed(sorted([(loss, i) for i, loss in enumerate(losses)])))
top_loss_idxs = [sl[1] for sl in sorted_losses]
min_losses = [sl[1] for sl in reversed(sorted_losses)]

In [ ]:
idx = top_loss_idxs[19]
viz, img, mo = show_gradcam_for_image(
    learner,
    learner.dls.valid_ds[idx][0].unsqueeze(0),
    1,
    "0.7.1.convpath.1.0",
)
print(mo)
show([viz, img])


In [ ]:
torch.save(
    learner.model.state_dict(),
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/baseline-random-ft.pth",
)

The model is now looking at the right places, train for full batch for some time

In [ ]:
dls = get_dls(20000, random_tfm)
learner.dls = dls

In [ ]:
learner.freeze()
learner.fit_one_cycle(5)
learner.unfreeze()
learner.fit_one_cycle(10)

In [ ]:
torch.save(
    learner.model.state_dict(),
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/baseline-random-ft-on-all-data-iter-10.pth",
)

In [ ]:
st_ed_tfm = partial(step_down_tfm, ratio=0.5)

In [ ]:
st_ed_tfm_50 = partial(st_ed_tfm, add_noise_chance=0.5)
st_ed_tfm_15 = partial(st_ed_tfm, add_noise_chance=0.15)
st_ed_tfm_0 = partial(st_ed_tfm, add_noise_chance=-1)

In [ ]:
dls = get_dls(2000, st_ed_tfm_50)
learner.dls = dls
learner.freeze()
learner.fit_one_cycle(3)
learner.unfreeze()
learner.fit_one_cycle(5)

In [ ]:
dls = get_dls(2000, st_ed_tfm_15)
learner.dls = dls
learner.freeze()
learner.fit_one_cycle(3)
learner.unfreeze()
learner.fit_one_cycle(5)

In [ ]:
torch.save(
    learner.model.state_dict(),
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/st_ed_tfm15-iter-5.pth",
)

In [ ]:
dls = get_dls(2000, st_ed_tfm_0)
learner.dls = dls
learner.freeze()
learner.fit_one_cycle(3)
learner.unfreeze()
learner.fit_one_cycle(5)

In [ ]:
torch.save(
    learner.model.state_dict(),
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/st_ed_tfm0-iter-5.pth",
)

In [ ]:
dls = get_dls(20000, st_ed_tfm_0)
learner.dls = dls
learner.freeze()
learner.fit_one_cycle(3)
learner.unfreeze()
learner.fit_one_cycle(15)

In [ ]:
torch.save(
    learner.model.state_dict(),
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/st_ed_tfm0-final-all-data-iter-20.pth",
)

In [ ]:
learner

# Train for small size (smallnet 64x64)

In [ ]:
st_ed_learner = get_initialised_learner()

# NEGMASK_224_STEP_EDGE = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/st_ed_tfm0-final-all-data-iter-20.pth"
# NEG_MASKSTEP_EDGE_MODEL_PATH = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-unblur/tfm-stepdown_ratio-5_samples-all-xresnet18_iter-20.pth"
NEGMASK_224_STEP_EDGE = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/tfm-stepedge_data-withusefultaco_iter-35_arch-xresnet18.pth"
# PHASE_2_STEP_EDGE_MODEL_PATH = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-unblur/tfm-stepdown_ratio-5_samples-all-xresnet18_iter-20.pth"
state_dict = torch.load(NEGMASK_224_STEP_EDGE, default_device())
st_ed_learner.load_state_dict(state_dict)

In [ ]:
35 * 35

In [ ]:
from mtrain.example_dir.learners import step_downer

dls = get_dls(
    20000, step_downer, 224, DS_PATH, min_area=2, min_bbox_length=1, max_area=1000
)

In [ ]:
st_ed_learner.dls = dls

In [ ]:
len(dls.train_ds), len(dls.valid_ds)

In [ ]:
vis_sample(dls, 22)

In [ ]:
st_ed_learner.freeze()
st_ed_learner.fit_one_cycle(3)

In [ ]:
res = show_reports(st_ed_learner)

In [ ]:
torch.save(
    st_ed_learner.model.state_dict(),
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/sm-st_ed_tfm0-final-all-data-with-taco-iter-10.pth",
)

In [ ]:
st_ed_learner.unfreeze()
st_ed_learner.fit_one_cycle(5)

In [ ]:
st_ed_learner.fit_one_cycle(15)

In [ ]:
st_ed_learner.lr_find()

In [ ]:
st_ed_learner.fit_one_cycle(5, lr_max=slice(1e-5, 1e-4))